<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/04_portfolio_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 04 · Portfolio Optimization — Exploratory & Didactic

**Data Science · Università Cattolica del Sacro Cuore, Milan**
Instructor: *Paolo Balasso*

---

## What you will explore

A guided exploration of portfolio construction with market data, focused on **understanding** rather than production deployment.

1. **Market data** — pull 8 ETFs across asset classes, inspect, visualise
2. **Returns** — daily returns, distributions, fat tails, arithmetic mean vs CAGR
3. **Correlations** — diversification regimes and the 2022 equity-bond flip
4. **Macro variables** — FRED data (GDP, CPI, Unemployment, 10y yield, USREC), YoY transformations, publication lags
5. **Investment Clock** — a level-based simplification with 4 phases (Reflation / Recovery / Overheat / Stagflation)
6. **Conditional analysis** — what worked in each regime, and how thin the evidence is
7. **Seven portfolios** — Equal-Weight, Min-Vol, Max-Sharpe, Max-Sharpe with a 15% cap, Permanent-style (Browne), All-Weather-style, and the 60/40 benchmark
8. **In-sample vs out-of-sample** — fit on 2014-2019, test on 2020-2024
9. **Regime-aware tilt** — an additive tilt on Equal-Weight, tested walk-forward
10. **Caveats** — look-ahead bias, in-sample optimism, small samples, publication lag
11. **Discussion & references**

## Data: live first, offline fallback second

The notebook first tries to download live data: Yahoo Finance via `yfinance`, and FRED via its public CSV endpoint. If a download fails or comes back incomplete, it switches **automatically** to a clearly labelled **synthetic** dataset (Section 0), so every cell still runs. A banner tells you which source is in use, and charts built on synthetic data say so in the title.

Synthetic numbers are realistic in shape, but they are **not** market history. Never quote them as facts.

## Approach

Short, readable functions. Lots of plots. Explicit data-science hygiene checks (shapes, NaN, duplicates, train/test awareness).
**Not** a production backtest. The goal is *content sedimentation*: making the concepts stick.

> *Past performance is not indicative of future results. This notebook is for educational purposes only. Nothing here constitutes investment advice.*

---

## 0 · Setup

In [ ]:
# Colab-friendly install: only what is missing (Colab already ships pandas, scipy, matplotlib, seaborn)
try:
    import yfinance  # noqa: F401
except ImportError:
    %pip install -q "yfinance>=0.2.40"
# If the market banner in Section 1 says SYNTHETIC on Colab, an outdated yfinance is a common cause:
# run  %pip install -q -U yfinance  then Runtime > Restart session, and run all again.

In [ ]:
import os, io, urllib.request, warnings
warnings.filterwarnings('ignore', category=FutureWarning)   # silence only noisy deprecations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

try:
    import yfinance as yf
except Exception as e:                  # no yfinance -> the offline fallback takes over
    yf = None
    print('yfinance not available:', type(e).__name__)

# Course palette
NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})

TRADING_DAYS = 252
rf_annual = 0.016   # risk-free rate for Sharpe: the 3-month T-bill averaged ~1.6% over 2014-2024
                    # (FRED TB3MS; Section 4 re-checks it on live data). Not the 10y yield (~2.5%):
                    # Sharpe needs a short, (almost) riskless rate.

# 'auto' = try live data, fall back to synthetic | 'offline' = synthetic only | 'live' = fail loudly
DATA_MODE = os.environ.get('NB04_DATA_MODE', 'auto').lower()
SEED = 42
print(f'pandas {pd.__version__} | numpy {np.__version__} | data mode: {DATA_MODE}')
print('Setup complete.')

### Offline fallback — synthetic market and macro data

Live downloads fail more often than you think: rate limits when a whole class pulls Yahoo at the same time, a corporate proxy, an SSL certificate problem. The next cell defines two generators. They run **only** if a download fails or returns incomplete data:

- `make_synthetic_market` — daily prices for the same 8 tickers, 2014-2024, on a US trading calendar. Mean, volatility and correlation change by sub-period (2014-19, 2020-21, 2022, 2023-24) to mimic the broad behaviour of each asset class, including the equity-bond correlation turning positive in 2022. Student-t shocks, volatility clustering and occasional crash days give fat tails and negative equity skew.
- `make_synthetic_macro` — the five FRED-shaped series from 1990: stylised paths for real GDP growth, core inflation, unemployment and the 10y yield, plus the NBER recession months.

Both are **simulated, not real data**, and they are not a copy of any vendor's prices. You can run the notebook without reading this cell.

In [ ]:
# ---- Offline fallback generators (used only when live data fails) -------------------
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

def banner(kind, live, detail=''):
    """Print which data source is in use, impossible to miss."""
    line = '=' * 80
    print(line)
    if live:
        print(f' DATA SOURCE ({kind}): LIVE download. {detail}')
    else:
        print(f' DATA SOURCE ({kind}): OFFLINE SYNTHETIC FALLBACK. {detail}')
        print(' Simulated data, NOT real history: the shapes are realistic, the numbers are not facts.')
    print(line)

def _valid_corr(C):
    """Clip negative eigenvalues so that C is a valid correlation matrix."""
    vals, vecs = np.linalg.eigh(C)
    C = vecs @ np.diag(np.clip(vals, 1e-4, None)) @ vecs.T
    d = np.sqrt(np.diag(C))
    return C / np.outer(d, d)

def make_synthetic_market(tickers, start, end, seed=SEED):
    """Simulated daily prices (start = 100) with sub-period drift, vol and correlation."""
    names = ['SPY', 'QQQ', 'IWM', 'EFA', 'TLT', 'IEF', 'GLD', 'DBC']
    unknown = [t for t in tickers if t not in names]
    if unknown:
        raise ValueError(f'The offline fallback only simulates {names}: no synthetic series for {unknown}.')
    rng = np.random.default_rng(seed)
    days = pd.date_range(start, end, freq=CustomBusinessDay(calendar=USFederalHolidayCalendar()))
    # Stylised annual mean and vol per sub-period (order = names), plus the equity-bond correlation
    segs = [('2014-01-01', '2019-12-31', [.12, .17, .09, .05, .07, .035, .04, -.06],
             [.13, .17, .16, .14, .12, .05, .13, .15], -0.40),
            ('2020-01-01', '2021-12-31', [.22, .33, .20, .11, .07, .03, .10, .15],
             [.25, .28, .32, .24, .18, .065, .17, .21], -0.35),
            ('2022-01-01', '2022-12-31', [-.17, -.33, -.19, -.13, -.33, -.15, .00, .21],
             [.24, .32, .28, .22, .20, .10, .15, .26], 0.15),
            ('2023-01-01', '2024-12-31', [.23, .33, .15, .11, -.01, .02, .18, -.01],
             [.13, .18, .20, .13, .16, .08, .14, .15], 0.10)]
    segs[0] = (min(str(start), segs[0][0]),) + segs[0][1:]                  # stretch the first and last
    segs[-1] = segs[-1][:1] + (max(str(end), segs[-1][1]),) + segs[-1][2:]  # sub-period to START / END
    base = np.array([   # long-run correlations; the equity-bond block is overwritten per period
        [1.00,  .93,  .86,  .85,  0.0,  0.0,  .03,  .34],
        [ .93, 1.00,  .77,  .76,  0.0,  0.0,  .04,  .26],
        [ .86,  .77, 1.00,  .79,  0.0,  0.0,  .04,  .34],
        [ .85,  .76,  .79, 1.00,  0.0,  0.0,  .11,  .38],
        [ 0.0,  0.0,  0.0,  0.0, 1.00,  .92,  .30, -.15],
        [ 0.0,  0.0,  0.0,  0.0,  .92, 1.00,  .38, -.10],
        [ .03,  .04,  .04,  .11,  .30,  .38, 1.00,  .26],
        [ .34,  .26,  .34,  .38, -.15, -.10,  .26, 1.00]])
    crash_load = np.array([1.0, 1.15, 1.2, 1.0, -0.35, -0.12, 0.10, 0.50])  # exposure to crash days
    df_t, h, chunks = 5, 0.0, []
    for a, b, mu, vol, rho in segs:
        idx = days[(days >= a) & (days <= b)]
        n = len(idx)
        C = base.copy()
        C[:4, 4] = C[4, :4] = rho
        C[:4, 5] = C[5, :4] = 0.9 * rho
        z = rng.standard_normal((n, 8)) @ np.linalg.cholesky(_valid_corr(C)).T
        z *= np.sqrt((df_t - 2) / rng.chisquare(df_t, size=(n, 1)))      # Student-t: fat tails
        hs = np.empty(n)
        for i in range(n):                                                # volatility clustering
            h = 0.97 * h + 0.15 * rng.standard_normal()
            hs[i] = h
        z *= np.exp(0.5 * hs)[:, None]
        z += np.outer((rng.random(n) < 0.01) * rng.normal(-3.5, 1.0, n), crash_load)  # crash days
        if a.startswith('2020'):                                          # a COVID-like sell-off
            z[(idx >= '2020-02-20') & (idx <= '2020-03-23')] -= 1.0 * crash_load
        # match each sub-period's mean and vol exactly (the shape of the shocks is kept)
        x = (z - z.mean(0)) / z.std(0) * (np.array(vol) / np.sqrt(TRADING_DAYS)) \
            + np.array(mu) / TRADING_DAYS
        chunks.append(pd.DataFrame(x, index=idx, columns=names))
    r = pd.concat(chunks).clip(lower=-0.5)
    return (100 * (1 + r).cumprod())[list(tickers)]

NBER_RECESSIONS = [('1990-07', '1991-03'), ('2001-03', '2001-11'),
                   ('2007-12', '2009-06'), ('2020-02', '2020-04')]

def _path(anchors, index):
    """Linear interpolation of {date: value} anchors onto a DatetimeIndex."""
    s = pd.Series(anchors, dtype=float)
    s.index = pd.to_datetime(s.index)
    full = s.reindex(s.index.union(index)).interpolate('time').ffill().bfill()
    return full.reindex(index).values

def _ar1(rng, n, phi, sd):
    e = np.zeros(n)
    for i in range(1, n):
        e[i] = phi * e[i - 1] + sd * rng.standard_normal()
    return e

def make_synthetic_macro(start, end, seed=SEED):
    """Stylised FRED-shaped series (approximate anchors + noise). NOT real FRED data."""
    rng = np.random.default_rng(seed + 1)
    months = pd.date_range(start, end, freq='MS')
    quarters = pd.date_range(start, end, freq='QS')
    bdays = pd.bdate_range(start, end)
    # Real GDP: YoY growth path (%) -> level series whose YoY follows the path
    g = _path({'1990-01': 1.8, '1991-01': -1.0, '1992-01': 2.5, '1994-01': 3.5, '1997-01': 4.0,
               '1999-01': 4.6, '2000-07': 4.0, '2001-10': 0.3, '2003-01': 2.0, '2004-07': 3.8,
               '2006-01': 3.0, '2007-07': 2.0, '2008-07': 0.5, '2009-04': -4.0, '2010-07': 2.7,
               '2012-01': 2.0, '2014-01': 1.5, '2015-04': 3.3, '2016-07': 1.5, '2018-07': 3.0,
               '2019-07': 2.2, '2020-01': 1.0, '2020-04': -7.5, '2020-10': -1.5, '2021-04': 12.0,
               '2021-10': 5.5, '2022-07': 1.8, '2023-07': 3.0, '2024-10': 2.5}, quarters) / 100
    g = g + _ar1(rng, len(g), 0.6, 0.003)
    lvl = np.empty(len(g))
    lvl[:4] = 9000 * 1.006 ** np.arange(4)
    for i in range(4, len(g)):
        lvl[i] = lvl[i - 4] * (1 + g[i])
    # Core CPI: YoY path (%) -> index level
    c = _path({'1990-01': 4.4, '1991-01': 5.0, '1992-01': 3.9, '1993-06': 3.3, '1995-01': 2.9,
               '1997-01': 2.5, '1999-01': 2.2, '2001-06': 2.6, '2003-12': 1.1, '2006-09': 2.9,
               '2008-06': 2.4, '2010-10': 0.6, '2012-04': 2.3, '2015-01': 1.6, '2016-06': 2.2,
               '2018-07': 2.4, '2020-06': 1.2, '2021-03': 1.6, '2021-06': 4.5, '2022-09': 6.6,
               '2023-06': 4.8, '2024-03': 3.8, '2024-12': 3.2}, months) / 100
    c = c + _ar1(rng, len(c), 0.7, 0.0012)
    cl = np.empty(len(c))
    cl[:12] = 130 * (1 + c[0]) ** (np.arange(12) / 12)
    for i in range(12, len(c)):
        cl[i] = cl[i - 12] * (1 + c[i])
    u = _path({'1990-01': 5.4, '1992-06': 7.8, '2000-04': 3.8, '2003-06': 6.3, '2007-03': 4.4,
               '2009-10': 10.0, '2015-01': 5.6, '2019-12': 3.6, '2020-03': 4.4, '2020-04': 14.8,
               '2020-12': 6.7, '2022-01': 4.0, '2022-12': 3.5, '2024-12': 4.2}, months)
    y = _path({'1990-01': 8.2, '1990-09': 8.9, '1993-10': 5.3, '1994-11': 7.9, '1998-10': 4.5,
               '2000-01': 6.6, '2003-06': 3.3, '2007-06': 5.1, '2008-12': 2.4, '2010-04': 3.9,
               '2012-07': 1.5, '2013-12': 3.0, '2016-07': 1.4, '2018-10': 3.2, '2020-08': 0.6,
               '2021-12': 1.5, '2022-10': 4.0, '2023-10': 4.9, '2024-09': 3.7, '2024-12': 4.4}, bdays)
    rec = np.zeros(len(months), dtype=int)
    for a, b in NBER_RECESSIONS:
        rec[(months >= a) & (months <= b)] = 1
    return {'GDPC1':    pd.DataFrame({'GDPC1': lvl}, index=quarters),
            'CPILFESL': pd.DataFrame({'CPILFESL': cl}, index=months),
            'UNRATE':   pd.DataFrame({'UNRATE': np.round(u + _ar1(rng, len(u), 0.8, 0.08), 1)}, index=months),
            'DGS10':    pd.DataFrame({'DGS10': np.round(y + _ar1(rng, len(y), 0.98, 0.03), 2)}, index=bdays),
            'USREC':    pd.DataFrame({'USREC': rec}, index=months)}

print('Offline fallback generators ready (used only if a live download fails).')

## 1 · The investable universe

We pick **8 ETFs** across the main asset classes — small enough to inspect manually, broad enough to show diversification.

Then we download the prices and **check** them. Watch the hygiene cell: `not prices.isna().any().any()` is also true for an *empty* table, so a failed download would pass it. Shape checks are what catch that.

In [ ]:
# 8-ETF universe — diversified across asset classes
TICKERS = {
    'SPY': 'US Large-Cap (S&P 500)',
    'QQQ': 'US Tech (Nasdaq 100)',
    'IWM': 'US Small-Cap (Russell 2000)',
    'EFA': 'Developed Intl ex-US',
    'TLT': 'US Long Treasury',
    'IEF': 'US Intermediate Treasury',
    'GLD': 'Gold',
    'DBC': 'Broad Commodities',
}
START, END = '2014-01-01', '2024-12-31'
print(f'{len(TICKERS)} ETFs across 5 asset classes')
for t, name in TICKERS.items():
    print(f'  {t:<5}  {name}')

In [ ]:
# Download adjusted closes (live), validate, and fall back to synthetic data if anything is off
def load_market_live(tickers, start, end):
    if yf is None:
        raise RuntimeError('yfinance not installed')
    import logging
    logging.getLogger('yfinance').setLevel(logging.CRITICAL)   # keep failures quiet: the banner reports them
    data = yf.download(tickers, start=start, end=end, auto_adjust=True,
                       progress=False, timeout=20)
    close = data['Close'] if isinstance(data.columns, pd.MultiIndex) else data
    return close.reindex(columns=tickers)

def market_ok(raw, min_days=2000, min_coverage=0.95):
    """Usable only if long enough AND every ticker is (almost) complete."""
    rows = 0 if raw is None else len(raw.dropna(how='all'))
    if rows < min_days:
        return False, f'only {rows} rows'
    cover = raw.dropna(how='all').notna().mean()
    bad = list(cover[cover < min_coverage].index)
    return (not bad), (f'incomplete tickers {bad}' if bad else 'ok')

tick_list = list(TICKERS)
raw, MARKET_LIVE, reason = None, False, 'offline mode requested'
if DATA_MODE != 'offline':
    try:
        print(f'Downloading {len(tick_list)} ETFs from Yahoo Finance (normally a few seconds; on a blocked '
              'network it can take a few minutes before the synthetic fallback starts)...', flush=True)
        raw = load_market_live(tick_list, START, END)
        MARKET_LIVE, reason = market_ok(raw)
    except Exception as e:
        reason = f'{type(e).__name__}: {str(e)[:80]}'
    if not MARKET_LIVE and DATA_MODE == 'live':
        raise RuntimeError(f'Live market data unusable ({reason}) and DATA_MODE = "live".')
if not MARKET_LIVE:
    raw = make_synthetic_market(tick_list, START, END)
MKT_TAG = '' if MARKET_LIVE else '  [SYNTHETIC DATA]'

banner('market', MARKET_LIVE, 'Yahoo Finance via yfinance.' if MARKET_LIVE
       else f'Live download not usable: {reason}.')
print('Raw shape (before cleaning):', raw.shape)
print('Missing values per ticker:')
print(raw.isna().sum().to_string())

In [ ]:
# DS-hygiene: forward-fill small gaps (holidays), drop leading/trailing NaN, dedup
prices = raw.dropna(how='all').ffill().dropna()
assert not prices.index.duplicated().any(), 'duplicate dates!'
assert not prices.isna().any().any(),       'residual NaN!'
# The two asserts above also pass on an EMPTY table. Shape checks catch a failed or partial download:
assert prices.shape[1] == len(TICKERS),     f'expected {len(TICKERS)} assets, got {prices.shape[1]}'
assert len(prices) > 2000,                  f'only {len(prices)} days: download incomplete?'
assert prices.index.is_monotonic_increasing, 'dates not sorted!'
print(f'Clean price matrix: {prices.shape[0]} days x {prices.shape[1]} assets')
print(f'From {prices.index.min().date()} to {prices.index.max().date()}')
prices.head()

In [ ]:
# Sanity-check: cumulative wealth (start=1) on log scale
norm = prices / prices.iloc[0]
fig, ax = plt.subplots(figsize=(11, 5))
for t in norm.columns:
    ax.plot(norm.index, norm[t], lw=1.1, alpha=0.85, label=t)
ax.set_yscale('log')
ax.set_title('Cumulative wealth, 2014-2024 (log scale, start=1)' + MKT_TAG,
             fontweight='bold', color=INK)
ax.set_ylabel('Wealth multiple')
ax.legend(loc='upper left', ncol=4, fontsize=9)
plt.tight_layout(); plt.show()

## 2 · From prices to returns

Two definitions students need to internalise:

- **Simple return** $r_t = P_t / P_{t-1} - 1$ — *additive across assets* in a portfolio
- **Log return**    $\ell_t = \ln(P_t / P_{t-1})$ — *additive across time*, symmetric around zero

We use simple returns for portfolio math, log returns only for inspection.

In [ ]:
# Daily simple and log returns
rets = prices.pct_change().dropna()
log_rets = np.log(prices / prices.shift(1)).dropna()
print('Daily simple returns:', rets.shape)
print('Any NaN?',              rets.isna().any().any())
print('Any duplicate dates?',  rets.index.duplicated().any())
rets.describe().round(4)

In [ ]:
# Small-multiples histogram: daily return distribution per asset
fig, axes = plt.subplots(2, 4, figsize=(13, 6), sharex=True)
for ax, t in zip(axes.flat, rets.columns):
    ax.hist(rets[t]*100, bins=60, color=NAVY, alpha=0.75,
            edgecolor='white', linewidth=0.3)
    ax.axvline(0, color=GOLD, ls='--', lw=0.8)
    ax.set_title(t, fontweight='bold', color=INK, fontsize=10)
    ax.set_xlabel('Daily return (%)')
plt.suptitle('Daily return distributions — note the fat tails' + MKT_TAG,
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Summary stats: arithmetic annual return, CAGR, vol, Sharpe, skew, kurtosis, max DD
rf_daily = rf_annual / TRADING_DAYS
n_years = len(rets) / TRADING_DAYS
summary = pd.DataFrame({
    'Ann. Return': rets.mean() * TRADING_DAYS,                        # arithmetic mean x 252
    'CAGR':        (prices.iloc[-1] / prices.iloc[0]) ** (1 / n_years) - 1,   # what you compounded
    'Ann. Vol':    rets.std() * np.sqrt(TRADING_DAYS),
    'Sharpe':      (rets.mean() - rf_daily) / rets.std() * np.sqrt(TRADING_DAYS),
    'Skew':        rets.skew(),
    'Kurtosis':    rets.kurtosis(),                                    # excess kurtosis: Gaussian = 0
    'Max DD':      ((prices / prices.cummax()) - 1).min(),
}).round(3).sort_values('Sharpe', ascending=False)
summary

In [ ]:
# Read-outs computed from the table above: they stay true if the data change
equities, bonds = ['SPY', 'QQQ', 'IWM', 'EFA'], ['TLT', 'IEF']
rank = summary['Sharpe'].rank(ascending=False).astype(int)
print(f"Excess kurtosis > 0 for {(summary['Kurtosis'] > 0).sum()} of {len(summary)} assets.")
print('Equity skew:   ' + ', '.join(f"{t} {summary.loc[t, 'Skew']:+.2f}" for t in equities))
print(f'Sharpe rank of the Treasury ETFs (1 = best of {len(summary)}): '
      + ', '.join(f"{t} #{rank[t]} (Sharpe {summary.loc[t, 'Sharpe']:.2f})" for t in bonds))
print(f"Max drawdown:  TLT {summary.loc['TLT', 'Max DD']:.0%} vs SPY {summary.loc['SPY', 'Max DD']:.0%}")
gap = (summary['Ann. Return'] - summary['CAGR']).sort_values(ascending=False)
print(f'Largest gap arithmetic mean - CAGR: {gap.index[0]} {gap.iloc[0]:.1%} per year (volatility drag)')

**Read-outs:**
- Excess kurtosis > 0 for **all** assets — daily returns are **not** Gaussian.
- Equities show negative skew (more left-tail surprises).
- **Bonds did not pay in this window.** The Treasury ETFs sit near the bottom of the Sharpe ranking, and long bonds (TLT) went through an equity-sized drawdown in the 2022 rate shock. Their job in a portfolio is the (usually) negative correlation with equities, not their stand-alone Sharpe — and Section 3 shows when even that failed.
- `Ann. Return` is the arithmetic mean × 252; `CAGR` is what an investor actually compounded. The gap grows with volatility (*volatility drag*).
- Max drawdowns are non-trivial for everything: diversification ≠ free lunch.

## 3 · Correlations — the diversification story

Correlations are **not** constants. Below: full-sample heatmap, then a rolling-window view of the famous SPY-TLT pair.

In [ ]:
# Full-sample correlation heatmap
corr = rets.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Daily return correlations — full sample' + MKT_TAG,
             fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 60-day correlation: equities vs long bonds
roll = rets['SPY'].rolling(60).corr(rets['TLT']).dropna()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(roll.index, roll, color=NAVY, lw=1.2)
ax.axhline(0, color=MUTED, lw=0.7, ls='--')
ax.fill_between(roll.index, roll, 0, where=(roll < 0),
                color=GREEN, alpha=0.25, label='Diversifying')
ax.fill_between(roll.index, roll, 0, where=(roll >= 0),
                color=RED, alpha=0.25, label='Co-moving')
ax.set_title('Rolling 60-day correlation: SPY vs TLT' + MKT_TAG,
             fontweight='bold', color=INK)
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

# Year by year: average correlation and share of days with positive correlation
yearly = pd.DataFrame({'mean corr':  roll.groupby(roll.index.year).mean(),
                       '% days > 0': (roll > 0).groupby(roll.index.year).mean() * 100})
print(yearly.T.round(2).to_string())
before = (roll.loc[:'2020'] > 0).mean()
after = (roll.loc['2022':] > 0).mean()
print(f'\nPositive SPY-TLT correlation on {before:.0%} of days in 2014-2020 vs {after:.0%} in 2022-2024.')
print('When the correlation is positive, long bonds stop cushioning equity losses.')

## 4 · Macro variables from FRED

Markets do not float in a vacuum. We pull **five** macro series — four signals plus a recession overlay — from **FRED** (Federal Reserve Economic Data), the same source used by central banks and institutional research:

| Series      | What it is                              | Frequency | Transformation |
|-------------|------------------------------------------|-----------|----------------|
| `GDPC1`    | Real GDP                                  | Quarterly | YoY % change   |
| `CPILFESL` | Core CPI (excludes food & energy)        | Monthly   | YoY % change   |
| `UNRATE`   | Civilian unemployment rate                | Monthly   | Level          |
| `DGS10`    | 10-year Treasury yield                    | Daily     | Monthly avg    |
| `USREC`    | NBER recession indicator (0 / 1)         | Monthly   | Used as overlay|

**Why YoY?** Levels trend upward over decades. Comparing each value with the same period one year earlier removes the trend and the seasonality, and gives a growth rate we can compare with a threshold. `GDPC1` is *real* GDP, already net of inflation, so its YoY change is real growth.

**Publication lag.** A month's CPI is published around the middle of the following month. A quarter's GDP is first published about a month after the quarter ends — up to four months after the first month we label with it — and it is revised later. So we also build a *real-time* panel, `macro_rt`, that only uses what was known at each date. We use it wherever a decision is made (Section 9).

We read FRED's public CSV endpoint with plain `pandas` — no extra package. (`pandas_datareader` is no longer actively maintained, so we avoid depending on it.)

*Inspiration: Paolo Balasso's [YoutubeTrading/EconomicRegime](https://github.com/paolobalasso/YoutubeTrading/blob/main/EconomicRegime.ipynb).*

In [ ]:
# Pull macro series from FRED (1990 onward to span multiple regimes), with offline fallback
macro_start = '1990-01-01'
FRED_IDS = ['GDPC1', 'CPILFESL', 'UNRATE', 'DGS10', 'USREC']

def fred_series(sid, start, end, timeout=15):
    """One FRED series from the public CSV endpoint ('.' marks a missing value)."""
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}&cosd={start}&coed={end}'
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        text = resp.read().decode('utf-8')
    df = pd.read_csv(io.StringIO(text), index_col=0, parse_dates=True, na_values='.')
    df.columns = [sid]
    return df.loc[start:end].astype(float)

MAX_AGE_DAYS = {'GDPC1': 300}   # quarterly GDP: dated at the quarter start, published ~1 month after it ends

def macro_ok(frames, end):
    for sid in FRED_IDS:
        s = frames.get(sid, pd.DataFrame()).dropna()
        if s.empty:
            return False, f'{sid} is empty'
        if s.index.max() < pd.Timestamp(end) - pd.Timedelta(days=MAX_AGE_DAYS.get(sid, 120)):
            return False, f'{sid} stops at {s.index.max().date()}'
    return True, 'ok'

fred, MACRO_LIVE, reason = {}, False, 'offline mode requested'
if DATA_MODE != 'offline':
    try:
        for sid in FRED_IDS:                   # stops at the first failure
            fred[sid] = fred_series(sid, macro_start, END)
        MACRO_LIVE, reason = macro_ok(fred, END)
    except Exception as e:
        reason = f'{type(e).__name__}: {str(e)[:80]}'
    if not MACRO_LIVE and DATA_MODE == 'live':
        raise RuntimeError(f'Live FRED data unusable ({reason}) and DATA_MODE = "live".')
if not MACRO_LIVE:
    fred = make_synthetic_macro(macro_start, END)
MAC_TAG = '' if MACRO_LIVE else '  [SYNTHETIC DATA]'
gdp, cpi, une, ten, rec = (fred[s] for s in FRED_IDS)

banner('macro', MACRO_LIVE, 'FRED public CSV endpoint.' if MACRO_LIVE
       else f'Live download not usable: {reason}.')
print('Shapes:')
print('  GDPC1    (real GDP, quarterly):', gdp.shape)
print('  CPILFESL (core CPI, monthly):  ', cpi.shape)
print('  UNRATE   (unemployment):       ', une.shape)
print('  DGS10    (10y yield, daily):   ', ten.shape)
print('  USREC    (recession 0/1):      ', rec.shape)

# Check the risk-free rate of Section 0 against the data (live runs only)
if MACRO_LIVE:
    try:
        tb3 = fred_series('TB3MS', START, END)['TB3MS'] / 100
        print(f'\nCheck: 3-month T-bill (TB3MS) averaged {tb3.mean():.2%} over {START[:4]}-{END[:4]}; '
              f'rf_annual used for Sharpe = {rf_annual:.1%}.')
    except Exception as e:
        print(f'\nT-bill check skipped ({type(e).__name__}).')
else:
    print(f'\nT-bill check skipped (synthetic macro). rf_annual used for Sharpe = {rf_annual:.1%}.')

In [ ]:
# Transformations: YoY for GDP/CPI, monthly average for 10y yield
gdp_yoy = gdp.pct_change(4).dropna().rename(columns={'GDPC1': 'GDP_YoY'})
cpi_yoy = cpi.pct_change(12).dropna().rename(columns={'CPILFESL': 'CPI_YoY'})
ten_m   = ten.resample('ME').mean().rename(columns={'DGS10': 'TenY'})
une_m   = une.rename(columns={'UNRATE': 'Unemp'})
rec_m   = rec.rename(columns={'USREC': 'Rec'})

# Combine into a monthly macro panel — forward-fill quarterly GDP
macro = (pd.concat([gdp_yoy, cpi_yoy, une_m, ten_m, rec_m], axis=1, sort=True)
           .resample('ME').last().ffill().dropna())
print(f'Macro panel: {macro.shape[0]} months from {macro.index.min().date()} to {macro.index.max().date()}')

# Real-time panel: shift each signal by its publication lag (in months)
macro_rt = macro.copy()
macro_rt['GDP_YoY'] = macro['GDP_YoY'].shift(3)   # Q1 sits in Jan-Mar here; first published ~end of April
macro_rt['CPI_YoY'] = macro['CPI_YoY'].shift(1)   # month m is published mid-month m+1
macro_rt = macro_rt.dropna()
print(f'Real-time panel: {macro_rt.shape[0]} months (GDP lagged 3 months, CPI lagged 1 month)')
macro.tail()

In [ ]:
# 2x2 panel: visualise all four macro variables with NBER recession shading
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
panels = [
    (axes[0,0], macro['GDP_YoY']*100, 'Real GDP YoY (%)',      NAVY,  2.5),
    (axes[0,1], macro['CPI_YoY']*100, 'Core CPI YoY (%)',      RED,   3.0),
    (axes[1,0], macro['Unemp'],       'Unemployment rate (%)', GOLD,  None),
    (axes[1,1], macro['TenY'],        '10y Treasury yield (%)', GREEN, None),
]
for ax, series, title, color, threshold in panels:
    ax.plot(series.index, series, color=color, lw=1.2)
    if threshold is not None:
        ax.axhline(threshold, color=MUTED, ls='--', lw=0.7,
                   label=f'{threshold}% threshold')
        ax.legend(loc='upper right', fontsize=8)
    for dt, val in rec.itertuples():
        if val == 1:
            ax.axvspan(dt, dt + pd.Timedelta(days=31),
                       color='lightgray', alpha=0.4)
    ax.set_title(title, fontweight='bold', color=INK)
plt.suptitle('US Macro Panel — grey bands = NBER-dated recessions' + MAC_TAG,
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

## 5 · Investment Clock — a level-based simplification

The **Investment Clock** (Greetham & Hartnett, Merrill Lynch, 2004) places the economy in one of four phases using two signals: **growth** relative to its trend and **inflation**, each judged by its *direction* (rising or falling). The phases tend to follow each other, like the hours on a clock.

Here we use a **simpler, level-based** version: growth is "high" when real GDP YoY is above 2.5%, inflation is "high" when core CPI YoY is above 3.0%.

| Phase           | Growth | Inflation | Assets that *historically* worked  |
|-----------------|--------|-----------|--------------------------------------|
| **Reflation**  | low    | low       | Long-duration bonds, defensives     |
| **Recovery**   | high   | low       | Equities (cyclicals), credit        |
| **Overheat**   | high   | high      | Commodities, real assets            |
| **Stagflation**| low    | high      | Cash, gold, short-duration          |

**Why the simplification?** Every student can trace exactly how a month gets its label. **The price:** levels move slowly. Core CPI stayed below 3% for most of the 2010s, so in our 2014-2024 return window the high-inflation phases cluster in 2021-2024. The regimes then partly work as **period labels** (before vs after the 2021 inflation surge), not as repeated macro states. Section 6 makes this visible.

In [ ]:
REGIMES = ['Reflation', 'Recovery', 'Overheat', 'Stagflation']
GDP_CUT, CPI_CUT = 0.025, 0.030   # try 0.020 / 0.025, then re-run from this cell down

def investment_clock(df, gdp_cut=GDP_CUT, cpi_cut=CPI_CUT):
    """Classify each row into one of four macro regimes (level-based)."""
    out = df.copy()
    out['Growth']    = np.where(out['GDP_YoY'] > gdp_cut, 'high', 'low')
    out['Inflation'] = np.where(out['CPI_YoY'] > cpi_cut, 'high', 'low')
    conds = [
        (out['Growth']=='low')  & (out['Inflation']=='low'),
        (out['Growth']=='high') & (out['Inflation']=='low'),
        (out['Growth']=='high') & (out['Inflation']=='high'),
        (out['Growth']=='low')  & (out['Inflation']=='high'),
    ]
    out['Regime'] = np.select(conds, REGIMES, default='Unknown')
    return out

clk = investment_clock(macro)
regime_colors = {
    'Reflation':   '#9CA3AF',
    'Recovery':    '#2D6E3E',
    'Overheat':    '#D4A017',
    'Stagflation': '#B32D2D',
}
last = clk.iloc[-1]
print(f"Latest regime in sample ({clk.index[-1].date()}): {last['Regime']}")
print(f"  real GDP YoY {last['GDP_YoY']:.2%} (cut {GDP_CUT:.2%}) | "
      f"core CPI YoY {last['CPI_YoY']:.2%} (cut {CPI_CUT:.2%})")   # 2 decimals: borderline cases
print('  For today\'s label, re-run with END = today: the latest GDP quarter will still be revised.')
SPAN = f'{clk.index[0].year}-{clk.index[-1].year}'
print(f'\nRegime frequency ({SPAN}, months):')
print(clk['Regime'].value_counts().reindex(REGIMES).fillna(0).astype(int).to_string())

In [ ]:
# Visualise: GDP & CPI YoY coloured by current regime + pie of regime frequency
fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={'width_ratios':[2,1]})

axes[0].plot(clk.index, clk['GDP_YoY']*100, color=NAVY, lw=1.0, label='GDP YoY')
axes[0].plot(clk.index, clk['CPI_YoY']*100, color=RED,  lw=1.0, label='CPI YoY')
ymin, ymax = -5, 12
for phase, col in regime_colors.items():
    mask = (clk['Regime'] == phase).values
    axes[0].fill_between(clk.index, ymin, ymax, where=mask,
                          color=col, alpha=0.18, label=phase)
axes[0].set_ylim(ymin, ymax); axes[0].set_ylabel('YoY (%)')
axes[0].set_title('Investment Clock (level-based) — phase classification' + MAC_TAG,
                   fontweight='bold', color=INK)
axes[0].legend(loc='upper right', ncol=2, fontsize=8)

counts = clk['Regime'].value_counts()
axes[1].pie(counts.values, labels=counts.index,
             colors=[regime_colors[k] for k in counts.index],
             autopct='%.0f%%', startangle=90,
             wedgeprops={'edgecolor':'white','linewidth':1})
axes[1].set_title(f'Regime frequency ({SPAN})',
                   fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

## 6 · What worked in each regime?

We compute the **average annualised return** of each ETF, **conditional on the macro regime** active that month. This is not a backtest — it is a historical conditional average. It uses the revised, same-month labels: fine for description, not for decisions.

Read the heatmap as a set of hypotheses, not as a playbook. First check how many months sit behind each row, and *when* they are.

In [ ]:
# Align our monthly returns with the regime panel
monthly_rets = (1 + rets).resample('ME').prod() - 1
clk_m = clk.resample('ME').last()
joined = monthly_rets.join(clk_m['Regime'], how='inner').dropna(subset=['Regime'])

# Average monthly return per regime per asset, annualised (x12)
regime_ret = joined.groupby('Regime').mean(numeric_only=True) * 12
regime_ret = regime_ret.reindex([r for r in REGIMES if r in regime_ret.index])

# How many months in each regime within OUR sample (2014-2024) — and WHEN?
when = joined.index.to_series()
spans = pd.DataFrame({
    'months':     when.groupby(joined['Regime']).count(),
    'first':      when.groupby(joined['Regime']).min().dt.strftime('%Y-%m'),
    'last':       when.groupby(joined['Regime']).max().dt.strftime('%Y-%m'),
    'share 2021+': (when >= '2021-01-01').groupby(joined['Regime']).mean().round(2),
}).reindex(REGIMES)
spans['months'] = spans['months'].fillna(0).astype(int)
print(f'Months per regime in {joined.index[0].year}-{joined.index[-1].year}, '
      'first/last month, share of months from 2021 on:')
print(spans.to_string())
print('\n[!] Regimes with <20 months are statistically thin. A regime that lives almost')
print('    only after 2021 mostly measures the 2021-2024 market, not a repeatable state.')
if MARKET_LIVE != MACRO_LIVE:
    print('[!] Returns and regimes come from different sources (one of them synthetic):')
    print('    the conditional returns below show the mechanics only, not a finding.')

In [ ]:
# Conditional-return heatmap
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(regime_ret*100, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            cbar_kws={'label':'Annualised return (%)'}, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Annualised return per asset, conditional on macro regime' + (MKT_TAG or MAC_TAG),
             fontweight='bold', color=INK)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

**Read-out.** Before reading any colour, read the `months` and `share 2021+` columns above. If a regime has only a handful of months, its row is an anecdote. If it lives almost entirely after 2021, its row mostly describes one episode — the inflation surge and the 2022 bond crash — rather than "what assets do in that kind of economy". That is why Section 9 tests the regime idea out of sample instead of trusting this table.

## 7 · Seven portfolios: optimisers, rules and the 60/40 benchmark

We build seven portfolios with **short, readable functions** — no PyPortfolioOpt black boxes, no HRP from scratch.

1. **Equal-Weight (1/N)** — DeMiguel-Garlappi-Uppal (2009) benchmark
2. **Min-Vol** — minimise portfolio variance, long-only, fully invested
3. **Max-Sharpe** — tangency portfolio, long-only, fully invested
4. **Max-Sharpe, 15% cap** — the same optimiser with $w_i \le 0.15$: the *Max Allocation* constraint on the slide *Portfolio Constraints & Regularisation*
5. **Permanent-style (Browne 1981)** — 25% stocks / 25% long bonds / 25% gold / 25% cash. Our universe has no T-bill ETF, so intermediate Treasuries (IEF) stand in for cash
6. **All-Weather-style** — 30% stocks / 40% long bonds / 15% intermediate bonds / 7.5% gold / 7.5% commodities. This is the widely circulated *retail simplification* of the risk-parity idea, with **fixed capital weights**. A true risk-parity portfolio sizes positions by their risk contribution
7. **60/40** — 60% SPY / 40% IEF: the benchmark that every strategy in the course has to beat

All Sharpe ratios use the T-bill rate set in Section 0 (`rf_annual`).

In [ ]:
# Annualised inputs
mu = rets.mean().values * TRADING_DAYS
Sigma = (rets.cov() * TRADING_DAYS).values
n = len(mu)
tickers = list(rets.columns)

def perf(w, mu=mu, Sigma=Sigma, rf=rf_annual):
    """Return (annualised return, vol, Sharpe) for weights w."""
    r = w @ mu
    v = np.sqrt(w @ Sigma @ w)
    s = (r - rf) / v if np.isfinite(v) and v > 0 else np.nan   # NaN, never a fake 0.0
    return r, v, s

def _solve(objective, n, w_max):
    """Long-only, fully invested SLSQP with an optional cap per asset."""
    if w_max * n < 1 - 1e-9:
        raise ValueError(f'A cap of {w_max:.1%} on {n} assets cannot add up to 100%: '
                         f'use a cap of at least {1/n:.1%}.')
    cons = ({'type':'eq', 'fun': lambda w: w.sum() - 1.0},)
    bnd  = tuple((0.0, w_max) for _ in range(n))
    res  = minimize(objective, np.repeat(1/n, n),
                    method='SLSQP', bounds=bnd, constraints=cons)
    w = np.clip(res.x, 0, None)
    w = w / w.sum()
    if not res.success:                  # never silent; stop only if the weights are unusable
        if not (np.isfinite(w).all() and (w <= w_max + 1e-4).all()):
            raise RuntimeError(f'optimiser failed: {res.message}')
        print(f'[!] Optimiser warning ({res.message}): the weights are feasible, check them before use.')
    return w

def solve_max_sharpe(mu, Sigma, rf=rf_annual, w_max=1.0):
    return _solve(lambda w: -perf(w, mu, Sigma, rf)[2], len(mu), w_max)

def solve_min_vol(Sigma, w_max=1.0):
    return _solve(lambda w: np.sqrt(w @ Sigma @ w), Sigma.shape[0], w_max)

def static_w(tickers, alloc):
    """Build a weight vector from a {ticker: fraction} dict; renormalise."""
    w = np.zeros(len(tickers))
    for t, f in alloc.items():
        if t in tickers:
            w[tickers.index(t)] = f
    return w / w.sum() if w.sum() > 0 else w

W_CAP = 0.15                        # try 0.25 or 0.40; below 1/8 = 12.5% no portfolio is possible
CAP = f'Max-Sharpe {W_CAP * 100:g}%'  # label of the capped portfolio
w_ew   = np.repeat(1/n, n)
w_mv   = solve_min_vol(Sigma)
w_ms   = solve_max_sharpe(mu, Sigma)
w_cap  = solve_max_sharpe(mu, Sigma, w_max=W_CAP)
w_perm = static_w(tickers, {'SPY':0.25, 'TLT':0.25, 'GLD':0.25, 'IEF':0.25})
w_aw   = static_w(tickers, {'SPY':0.30, 'TLT':0.40, 'IEF':0.15,
                             'GLD':0.075, 'DBC':0.075})
w_6040 = static_w(tickers, {'SPY':0.60, 'IEF':0.40})

PORTS = {'Equal-Wt': w_ew, 'Min-Vol': w_mv, 'Max-Sharpe': w_ms, CAP: w_cap,
         'Permanent': w_perm, 'All-Weather': w_aw, '60/40': w_6040}
scorecard = pd.DataFrame({k: perf(w) for k, w in PORTS.items()},
                         index=['Ann. Return', 'Ann. Vol', 'Sharpe']).round(3)
scorecard

In [ ]:
# Visualise the seven allocations side-by-side, then measure concentration
alloc_df = pd.DataFrame(PORTS, index=tickers)
fig, ax = plt.subplots(figsize=(12, 5))
alloc_df.plot(kind='bar', ax=ax,
              color=[MUTED, GREEN, GOLD, '#E9C46A', NAVY, RED, '#8FA3BF'],
              edgecolor='black', linewidth=0.4)
ax.set_title('Portfolio weights by strategy' + MKT_TAG, fontweight='bold', color=INK)
ax.set_ylabel('Weight')
plt.xticks(rotation=45)
ax.legend(loc='upper right', fontsize=9, ncol=2)
plt.tight_layout(); plt.show()

conc = pd.DataFrame({'Holdings > 1%':       (alloc_df > 0.01).sum(),
                     'Max weight':          alloc_df.max(),
                     'Effective N (1/HHI)': 1 / (alloc_df ** 2).sum()}).T.round(2)
print(conc.to_string())
eff_n = conc.loc['Effective N (1/HHI)']
print(f'\nMost diversified:  {eff_n.idxmax()} (effective N {eff_n.max():.1f} of {n})')
print(f'Most concentrated: {eff_n.idxmin()} (effective N {eff_n.min():.1f})')
print(f'Min-Vol puts {w_mv.max():.0%} in {tickers[int(w_mv.argmax())]}; '
      f'Max-Sharpe holds {(w_ms > 0.01).sum()} assets; '
      f'the {W_CAP * 100:g}% cap spreads it over {(w_cap > 0.01).sum()} assets.')

**Read-out.** *Both* optimisers concentrate: Min-Vol piles into the calmest asset, Max-Sharpe into the best past performers. Equal-Weight is the most diversified by construction (effective N = 8). Effective N = 1 / Σ wᵢ² reads as "how many equally weighted assets this portfolio behaves like".

The 15% cap is a constraint acting as **regularisation**: compare the two Max-Sharpe columns in the scorecard. It costs in-sample Sharpe and buys diversification. Whether that trade pays off on new data is the question of Section 8.

### The five backtest metrics, live

The slide *Walk-Forward Validation & Backtest Metrics* lists five numbers: CAGR, Sharpe, Sortino, Max Drawdown and Calmar. Here they are for the seven portfolios, computed on daily returns with weights rebalanced daily to target and no trading costs. For the optimisers this is still **in-sample**: they were fitted on these same years.

In [ ]:
def metrics(port_rets, rf=rf_annual, periods=TRADING_DAYS):
    """CAGR, Vol, Sharpe, Sortino, Max DD and Calmar for periodic portfolio returns."""
    r = pd.Series(port_rets).dropna()
    wealth = (1 + r).cumprod()
    cagr = wealth.iloc[-1] ** (periods / len(r)) - 1
    vol = r.std() * np.sqrt(periods)
    excess = r.mean() * periods - rf
    downside = np.sqrt((np.minimum(r - rf / periods, 0) ** 2).mean() * periods)
    mdd = (wealth / wealth.cummax() - 1).min()
    return pd.Series({'CAGR': cagr, 'Vol': vol, 'Sharpe': excess / vol,
                      'Sortino': excess / downside, 'Max DD': mdd, 'Calmar': cagr / abs(mdd)})

port_daily = pd.DataFrame({k: rets @ w for k, w in PORTS.items()})
backtest = port_daily.apply(metrics).round(3)
backtest

## 8 · In-sample vs out-of-sample — the optimiser's optimism, measured

Everything above was fitted and scored on the **same** 2014-2024 data. Now we fit the optimisers on **2014-2019 only**, freeze the weights, and score every portfolio on **2020-2024**: data the optimiser never saw. The rule-based portfolios have no fitting step, so their weights do not change.

This is a single split, not a full walk-forward — but it is the first step of the procedure on the slide *Walk-Forward Validation & Backtest Metrics*.

In [ ]:
# Fit on 2014-2019, evaluate on 2020-2024
train, test = rets.loc[:'2019'], rets.loc['2020':]
mu_tr, S_tr = train.mean().values * TRADING_DAYS, (train.cov() * TRADING_DAYS).values
fitted = dict(PORTS)                                   # rule-based weights unchanged
fitted.update({'Min-Vol':        solve_min_vol(S_tr),
               'Max-Sharpe':     solve_max_sharpe(mu_tr, S_tr),
               CAP:              solve_max_sharpe(mu_tr, S_tr, w_max=W_CAP)})

print('Weights fitted on 2014-2019 only:')
print(pd.DataFrame({k: fitted[k] for k in ['Min-Vol', 'Max-Sharpe', CAP]},
                   index=tickers).T.round(2).to_string())

rows = {}
for k, w in fitted.items():
    m_in, m_out = metrics(train @ w), metrics(test @ w)
    rows[k] = {'Sharpe 2014-19 (fit)': m_in['Sharpe'], 'Sharpe 2020-24 (test)': m_out['Sharpe'],
               'CAGR 2020-24': m_out['CAGR'], 'Max DD 2020-24': m_out['Max DD']}
split_tbl = pd.DataFrame(rows).T.round(3)
ms = split_tbl.loc['Max-Sharpe']
print(f"\nMax-Sharpe: Sharpe {ms.iloc[0]:.2f} on the fit window -> {ms.iloc[1]:.2f} on the test window.")
cap = split_tbl.loc[CAP]
print(f"{CAP} cap: Sharpe {cap.iloc[0]:.2f} on the fit window -> {cap.iloc[1]:.2f} on the test window.")
rank_fit = split_tbl.iloc[:, 0].rank(ascending=False).astype(int)
rank_test = split_tbl.iloc[:, 1].rank(ascending=False).astype(int)
print('Rank on fit window -> rank on test window: '
      + ', '.join(f'{k} {rank_fit[k]}->{rank_test[k]}' for k in split_tbl.index))
split_tbl

**Read-out.** The fitted Max-Sharpe portfolio looks best on the data it was fitted on, and loses much of that edge on new data: part of what the optimiser "found" was noise in the 2014-2019 means. Now compare the two Max-Sharpe rows: the capped version gives up Sharpe on the fit window — does it hold up better on the test window? That is what *regularisation* means in practice. Simple rules have nothing to overfit, which is why 1/N and 60/40 are such stubborn benchmarks: check the rank changes printed above. One split is one story, not a proof. A walk-forward repeats the exercise window after window.

## 9 · Regime-aware tilt — overlay macro information, then test it honestly

**Why not tilt Max-Sharpe and compare in-sample?** Max-Sharpe is, by definition, the long-only portfolio with the highest Sharpe on the full-sample means and covariances. Any other weights — tilted or not — score lower or equal on those same inputs. That comparison cannot be won, so it teaches nothing.

So we change two things.

**1. A baseline that holds every asset: Equal-Weight, with an additive tilt.** (A multiplicative tilt on Max-Sharpe could only re-mix the few assets it already holds.)

$$w_i^{\text{tilted}} \propto \max\Bigl(0,\; w_i^{\text{base}} + \tfrac{\theta}{N}\, z_i^{\text{regime}}\Bigr)$$

where $z_i^{\text{regime}}$ is the z-score of asset $i$'s average return in the regime vs its cross-regime mean (capped at ±2), $N$ is the number of assets and $\theta$ the tilt strength.

**2. An honest evaluation.** First the tempting number: in-sample, on the months that built the regime table. Then a **walk-forward test on 2020-2024**: each month we use only the regime label *known at that time* (real-time panel, plus one month so the label is known before the month starts) and a regime table estimated only on *past* months.

In [ ]:
THETA, MIN_MONTHS = 0.30, 6

def regime_tilt(base_w, regime_table, regime, theta=THETA, z_cap=2.0):
    """Additive tilt of base weights toward assets that did well in `regime`."""
    if regime not in regime_table.index or len(regime_table) < 2:
        return base_w.copy()                            # regime never seen -> no tilt
    z = (regime_table.loc[regime] - regime_table.mean()) / (regime_table.std() + 1e-9)
    z = z.clip(-z_cap, z_cap).reindex(tickers).fillna(0.0).values
    w = np.clip(base_w + theta * z / len(base_w), 0, None)
    return w / w.sum()

latest = clk['Regime'].iloc[-1]
w_tilted = regime_tilt(w_ew, regime_ret, latest)
print(f'Latest regime in sample ({clk.index[-1].date()}): {latest}')

# Tempting but IN-SAMPLE: score both portfolios on the months that built the regime table
reg_months = joined.loc[joined['Regime'] == latest, tickers]
if len(reg_months) >= 3:
    ins = pd.DataFrame({'Equal-Wt (base)':      metrics(reg_months @ w_ew, periods=12),
                        f'Tilted ({latest})':   metrics(reg_months @ w_tilted, periods=12)})
    print(f'IN-SAMPLE on the {len(reg_months)} {latest} months of {START[:4]}-{END[:4]} '
          '(the same months that built the table):')
    print(ins.loc[['CAGR', 'Vol', 'Sharpe']].round(3).to_string())
else:
    print(f'Only {len(reg_months)} {latest} months in {START[:4]}-{END[:4]}: too few to score.')

In [ ]:
# Visualise the tilt vs the baseline
cmp = pd.DataFrame({
    'Equal-Weight (base)':         w_ew,
    f'Regime-tilted ({latest})':   w_tilted,
}, index=tickers)
fig, ax = plt.subplots(figsize=(11, 4))
cmp.plot(kind='bar', ax=ax, color=[MUTED, GOLD],
         edgecolor='black', linewidth=0.4)
ax.set_title(f'Regime-aware tilt — latest phase in sample: {latest}' + (MKT_TAG or MAC_TAG),
             fontweight='bold', color=INK)
ax.set_ylabel('Weight')
plt.xticks(rotation=45)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()
print('\nInterpretation: every asset moves, up or down, according to how unusual its return')
print(f'was in {latest} months compared with the other regimes, not its full-sample mean.')

In [ ]:
# Walk-forward 2020-2024: real-time labels, regime table estimated on past months only
label_rt = investment_clock(macro_rt)['Regime'].shift(1)       # known BEFORE the month starts
panel = monthly_rets.join(label_rt.rename('Regime'), how='inner').dropna(subset=['Regime'])

records = []
for t in panel.loc['2020':].index:
    past = panel.loc[panel.index < t]
    seen = past['Regime'].value_counts()
    table = (past.groupby('Regime')[tickers].mean() * 12).loc[seen[seen >= MIN_MONTHS].index]
    regime = panel.at[t, 'Regime']
    w_t = regime_tilt(w_ew, table, regime)
    r_t = panel.loc[t, tickers].values.astype(float)
    records.append({'date': t, 'regime': regime,
                    'active': bool(regime in table.index and len(table) >= 2),
                    'base': r_t @ w_ew, 'tilt': r_t @ w_t})
wf = pd.DataFrame(records).set_index('date')

oos = pd.DataFrame({'Equal-Wt (base)': metrics(wf['base'], periods=12),
                    'Regime-tilted':   metrics(wf['tilt'], periods=12)}).round(3)
print(f"OUT-OF-SAMPLE walk-forward {wf.index[0]:%Y-%m} to {wf.index[-1]:%Y-%m}: {len(wf)} months, "
      f"tilt active in {wf['active'].sum()} of them")
print(oos.to_string())
gap = oos.loc['Sharpe', 'Regime-tilted'] - oos.loc['Sharpe', 'Equal-Wt (base)']
print(f'\nSharpe, tilt minus base: {gap:+.2f} over {len(wf)} months.')
print('Real-time regime labels used in the test window (months):')
print(wf['regime'].value_counts().reindex(REGIMES).fillna(0).astype(int).to_string())

**Read-out.** Put the two tables side by side. In-sample, on the months that built the regime table, the tilt is flattered: it was designed with hindsight on exactly those months. Out of sample — real-time labels, a table built only from the past — compare the Sharpe gap again: with about 60 months, a difference of a few hundredths, or even a few tenths, is not evidence either way. Notice also the months where the tilt is **inactive**: a regime that first appears in 2021 needs months of history before it can be traded.

That is the honest status of most regime playbooks: interpretable, useful as context, rarely a reliable source of extra return on their own.

## 10 · Caveats — what would break this in production

This notebook is **didactic** by design. Before anyone deploys real capital based on similar logic, they would have to address:

0. **Data source.** If a banner said *SYNTHETIC*, every number in that part of the notebook is simulated. Use such a run to learn the mechanics; quote numbers only from a live run.
1. **Look-ahead bias.** Sections 5-6 use revised, same-month macro labels. A quarter's GDP is first published about a month after the quarter ends — up to four months after the first month we label with it — and revisions can change a regime *after the fact*. Section 9 uses the real-time panel; a production system would use first-release (vintage) data, for example from ALFRED.
2. **In-sample optimism.** Section 8 shows the fitted optimum losing much of its edge out of sample. Transaction costs would widen the gap.
3. **Sample size.** ~35 years of macro data contain only a handful of full cycles, and our 2014-2024 return window covers even fewer. Conditional returns based on <20 months are exploratory.
4. **Hard thresholds hide gradients.** GDP at 2.4% and 0.5% are very different, yet both get labelled 'low' here.
5. **Non-stationarity.** The equity-bond correlation turned positive around 2022 and stayed there for a while; risk premia move with monetary policy; macro relationships are not laws of nature.
6. **Frictions.** Transaction costs, slippage, taxes, borrowing constraints all eat returns. None are modelled here: every portfolio is rebalanced for free.

Used as a **complement** to factor models and risk management — not a substitute — macro regime analysis adds interpretable, narrative-friendly context to a quantitative portfolio.

---

## 11 · Discussion & references

### What we learned

1. Daily returns are **fat-tailed** for every asset — Gaussian assumptions fail.
2. Equity-bond **correlations are not constant** — 2022 reminded everyone, and the change did not reverse immediately.
3. **Both optimisers concentrate** — Min-Vol in the calmest asset, Max-Sharpe in the past winners. Constraints such as a 15% cap act as regularisation.
4. **In-sample numbers flatter fitted portfolios.** Simple rules (1/N, 60/40) have nothing to overfit, which makes them hard benchmarks out of sample.
5. **Macro variables add context** that pure return-based models lack: YoY transformations and recession overlays make the market story readable. Publication lags decide what you could actually have known.
6. **Regime tilts** are intuitive and easy to explain — that is their strength. A fair test needs real-time labels and out-of-sample evaluation, and in a short window the evidence is thin.

### Where to go next

- Turn the single split of Section 8 into a rolling **walk-forward backtest** with realistic frictions.
- Replace the level cut-offs with the original **direction-based** clock (growth vs trend, inflation rising or falling), or with rolling z-scores.
- Add an **inverse-volatility** or true **risk-parity** portfolio (weights from risk contributions) next to the All-Weather-style rule.
- Add a **factor exposure analysis** (Fama-French 3-factor) on the tilted portfolio.
- Stress-test on the **COVID shock** (Feb-Apr 2020) or the **2022 bond rout**.

### References

- Markowitz, H. (1952). *Portfolio Selection*. Journal of Finance.
- Browne, H., Coxon, T. (1981). *Inflation-Proofing Your Investments*.
- DeMiguel, V., Garlappi, L., Uppal, R. (2009). *Optimal Versus Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?*. RFS.
- Greetham, T., Hartnett, M. (2004). *The Investment Clock*. Merrill Lynch.
- Balasso, P. *YoutubeTrading* — [github.com/paolobalasso/YoutubeTrading](https://github.com/paolobalasso/YoutubeTrading).

---

**End of notebook.** About a minute of runtime on Colab CPU, most of it the downloads; the synthetic fallback itself runs in seconds. The goal was *content sedimentation*, not deployment — every cell should be inspectable, every plot tells a story.